In [1]:
import csv
import math
from collections import defaultdict
from typing import Dict, List, Tuple, Optional

CSV_PATH = "ucl_2425_league_phase_results.csv"

=

GAMMA_MAX = 2.0   #maximum allowed gamma. at GAMMA_MAX=2: P(draw|d=1) stays above ~7% for any delta.


def load_matches(csv_path: str) -> List[dict]:
    matches = []
    with open(csv_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            matches.append({
                "matchday": int(row["matchday"]),
                "home": row["home"],
                "away": row["away"],
                "hg": int(row["home_goals"]),
                "ag": int(row["away_goals"]),
            })
    return matches


def outcome_points(gf: int, ga: int) -> int:
    if gf > ga: return 3
    if gf == ga: return 1
    return 0


def compute_standard_points(matches: List[dict]) -> Dict[str, int]:
    pts: Dict[str, int] = defaultdict(int)
    for m in matches:
        h, a = m["home"], m["away"]
        pts[h] += outcome_points(m["hg"], m["ag"])
        pts[a] += outcome_points(m["ag"], m["hg"])
    return dict(pts)


def sort_table(d: Dict[str, float]) -> List[Tuple[str, float]]:
    return sorted(d.items(), key=lambda x: (-x[1], x[0]))


#records: (i, j, y, sigma)
def build_directed_records(
    matches: List[dict],
) -> Tuple[List[str], Dict[str, int], List[Tuple[int, int, int, int]]]:
    team_set = set()
    for m in matches:
        team_set.add(m["home"]); team_set.add(m["away"])
    teams = sorted(team_set)
    idx = {t: k for k, t in enumerate(teams)}

    records = []
    for m in matches:
        h, a = m["home"], m["away"]
        hi, ai = idx[h], idx[a]
        yh = outcome_points(m["hg"], m["ag"])
        ya = outcome_points(m["ag"], m["hg"])
        records.append((hi, ai, yh, +1))
        records.append((ai, hi, ya, -1))
    return teams, idx, records


#reparameterisations
def softplus(x: float) -> float:
    if x > 30: return x
    return math.log(1.0 + math.exp(x))

def sigmoid(x: float) -> float:
    if x > 30: return 1.0
    if x < -30: return 0.0
    return 1.0 / (1.0 + math.exp(-x))

def delta_from_raw(delta_raw: float) -> float:
    return softplus(delta_raw)

def gamma_from_raw(gamma_raw: float) -> float:
    return GAMMA_MAX * sigmoid(gamma_raw)

def d_delta_d_raw(delta_raw: float) -> float:
    return sigmoid(delta_raw)

def d_gamma_d_raw(gamma_raw: float) -> float:
    g = sigmoid(gamma_raw)
    return GAMMA_MAX * g * (1.0 - g)

#softmax with linear draw utility
def softmax_probs(
    d_tilde: float, delta: float, gamma: float
) -> Tuple[float, float, float]:
    u_w = d_tilde
    u_l = -d_tilde
    u_d = delta - gamma * abs(d_tilde)
    m = max(u_w, u_d, u_l)
    ew = math.exp(u_w - m)
    ed = math.exp(u_d - m)
    el = math.exp(u_l - m)
    z = ew + ed + el
    return ew / z, ed / z, el / z



#gradients in raw parameter space
def compute_gradients(
    records: List[Tuple[int, int, int, int]],
    s: List[float],
    delta_raw: float,
    gamma_raw: float,
    h: float,
    lambda_s: float,
) -> Tuple[List[float], float, float, float, float]:
    delta = delta_from_raw(delta_raw)
    gamma = gamma_from_raw(gamma_raw)

    n = len(s)
    grad_s = [0.0] * n
    grad_delta_raw = 0.0
    grad_gamma_raw = 0.0
    grad_h = 0.0
    ll = 0.0

    for i, j, y, sigma in records:
        d = s[i] - s[j]
        d_tilde = d + sigma * h

        p_w, p_d, p_l = softmax_probs(d_tilde, delta, gamma)

        is_w = 1.0 if y == 3 else 0.0
        is_d = 1.0 if y == 1 else 0.0
        is_l = 1.0 if y == 0 else 0.0

        if y == 3:   ll += math.log(p_w)
        elif y == 1: ll += math.log(p_d)
        else:        ll += math.log(p_l)

        grad_delta_direct = (is_d - p_d)
        grad_gamma_direct = -abs(d_tilde) * (is_d - p_d)

        grad_delta_raw += grad_delta_direct * d_delta_d_raw(delta_raw)
        grad_gamma_raw += grad_gamma_direct * d_gamma_d_raw(gamma_raw)

        sign_d = 1.0 if d_tilde > 0 else (-1.0 if d_tilde < 0 else 0.0)
        g_d = (is_w - p_w) - (is_l - p_l) - gamma * sign_d * (is_d - p_d)

        grad_s[i] += g_d
        grad_s[j] -= g_d
        grad_h    += sigma * g_d

    ll -= (lambda_s / 2.0) * sum(sk * sk for sk in s)
    for k in range(n):
        grad_s[k] -= lambda_s * s[k]

    return grad_s, grad_delta_raw, grad_gamma_raw, grad_h, ll


def project_mean_zero(s: List[float]) -> None:
    mean = sum(s) / len(s)
    for k in range(len(s)):
        s[k] -= mean


#gradient ascent with cosine LR decay + best-param tracking
def fit_strength_model(
    records: List[Tuple[int, int, int, int]],
    n_teams: int,
    max_iters: int = 10000,
    tol: float = 1e-10,
    eta_s: float = 0.01,
    eta_scalar: float = 0.01,
    init_delta_raw: float = 0.0,
    init_gamma_raw: float = 0.0,
    init_h: float = 0.3,
    lambda_s: float = 0.01,
    lr_floor: float = 1e-4,
    fit_h: bool = True,
) -> Tuple[List[float], float, float, float, float, float, List[float]]:
    s = [0.0] * n_teams
    delta_raw = init_delta_raw
    gamma_raw = init_gamma_raw
    h = init_h if fit_h else 0.0
    project_mean_zero(s)

    best_ll = -math.inf
    best_s = s[:]
    best_delta_raw = delta_raw
    best_gamma_raw = gamma_raw
    best_h = h

    ll_history: List[float] = []
    prev_ll: Optional[float] = None

    for iteration in range(max_iters):
        decay = 0.5 * (1.0 + math.cos(math.pi * iteration / max_iters))
        lr = decay + lr_floor

        grad_s, grad_dr, grad_gr, grad_h, ll = compute_gradients(
            records, s, delta_raw, gamma_raw, h, lambda_s
        )

        ll_history.append(ll)

        if ll > best_ll:
            best_ll = ll
            best_s = s[:]
            best_delta_raw = delta_raw
            best_gamma_raw = gamma_raw
            best_h = h

        for k in range(n_teams):
            s[k] += eta_s * lr * grad_s[k]
        project_mean_zero(s)

        delta_raw += eta_scalar * lr * grad_dr
        gamma_raw += eta_scalar * lr * grad_gr

        if fit_h:
            h += eta_scalar * lr * grad_h
            if h < 0.0: h = 0.0

        if prev_ll is not None and abs(ll - prev_ll) < tol:
            break
        prev_ll = ll

    delta = delta_from_raw(best_delta_raw)
    gamma = gamma_from_raw(best_gamma_raw)
    return best_s, delta, gamma, best_h, best_delta_raw, best_gamma_raw, ll_history


#ranking utilities
def get_rank(scored: Dict[str, float]) -> Dict[str, int]:
    rows = sorted(scored.items(), key=lambda x: (-x[1], x[0]))
    return {team: r + 1 for r, (team, _) in enumerate(rows)}


def kendall_tau(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    c = d = 0
    for p in range(n):
        for q in range(p + 1, n):
            t1, t2 = teams[p], teams[q]
            prod = (rank_a[t1] - rank_a[t2]) * (rank_b[t1] - rank_b[t2])
            if prod > 0: c += 1
            elif prod < 0: d += 1
    return (c - d) / (n * (n - 1) // 2)


def spearman_rho(rank_a: Dict[str, int], rank_b: Dict[str, int]) -> float:
    teams = list(rank_a.keys())
    n = len(teams)
    d_sq = sum((rank_a[t] - rank_b[t]) ** 2 for t in teams)
    return 1.0 - (6.0 * d_sq) / (n * (n * n - 1))


def topk_overlap(rank_a: Dict[str, int], rank_b: Dict[str, int], k: int) -> int:
    top_a = {t for t, r in rank_a.items() if r <= k}
    top_b = {t for t, r in rank_b.items() if r <= k}
    return len(top_a & top_b)


def print_strength_table(teams, s, points, std_rank, title="Latent strength ranking"):
    rows = sorted(((teams[i], s[i]) for i in range(len(teams))), key=lambda x: (-x[1], x[0]))
    rank = {t: r + 1 for r, (t, _) in enumerate(rows)}
    print(title)
    print(f"{'Pos':>3}  {'Team':25}  {'Pts':>4}  {'s_hat':>10}  {'Delta':>6}")
    print("=" * 58)
    for pos, (team, val) in enumerate(rows, start=1):
        d = std_rank.get(team, pos) - rank.get(team, pos)
        print(f"{pos:>3}  {team:25}  {points.get(team,0):>4}  {val:>10.6f}  {d:>+6}")
    print()


def print_convergence(ll_history: List[float], best_ll: float) -> None:
    n = len(ll_history)
    best_iter = max(range(n), key=lambda i: ll_history[i])
    print("Convergence diagnostics")
    print(f"  Iterations:        {n}")
    print(f"  Initial LL:        {ll_history[0]:.4f}")
    print(f"  Best LL:           {best_ll:.4f}  (iter {best_iter})")
    print(f"  Final LL:          {ll_history[-1]:.4f}")
    if ll_history[-1] < best_ll - 0.01:
        print(f"  *** Oscillation corrected: {best_ll - ll_history[-1]:.4f} below best ***")
    milestones = sorted(set(min(int(n * f), n - 1) for f in [0.1, 0.25, 0.5, 0.75, 1.0]))
    print(f"\n  Progress:")
    for m in milestones:
        print(f"    iter {m:>5}: ll = {ll_history[m]:.6f}")
    print()


def print_draw_probability_table(delta: float, gamma: float, h: float) -> None:
    print(f"Draw probability profile  (delta={delta:.4f}, gamma={gamma:.4f})")
    print(f"  (Actual UCL 24/25 draw rate: 34.7% — expect P(draw|d=0) near this)")
    print(f"  {'d_tilde':>8}  {'P(win)':>8}  {'P(draw)':>8}  {'P(loss)':>8}")
    for d in [0.0, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]:
        pw, pd, pl = softmax_probs(d, delta, gamma)
        print(f"  {d:>8.2f}  {pw:>8.3f}  {pd:>8.3f}  {pl:>8.3f}")
    print()


def print_home_analysis(matches, teams, s, delta, gamma, h):
    home_pts: Dict[str, int] = defaultdict(int)
    away_pts: Dict[str, int] = defaultdict(int)
    hg_cnt:   Dict[str, int] = defaultdict(int)
    ag_cnt:   Dict[str, int] = defaultdict(int)
    for m in matches:
        ht, at = m["home"], m["away"]
        home_pts[ht] += outcome_points(m["hg"], m["ag"])
        away_pts[at] += outcome_points(m["ag"], m["hg"])
        hg_cnt[ht] += 1; ag_cnt[at] += 1

    p_h, p_d_h, p_a = softmax_probs(h, delta, gamma)
    p_n, p_d_n, _   = softmax_probs(0.0, delta, gamma)

    print("Home advantage analysis")
    print(f"  Fitted h = {h:.6f}")
    print(f"  Neutral venue (d=0):  P(win)={p_n:.3f}  P(draw)={p_d_n:.3f}  P(loss)={1-p_n-p_d_n:.3f}")
    print(f"  With home boost:      P(win)={p_h:.3f}  P(draw)={p_d_h:.3f}  P(loss)={p_a:.3f}")
    print()


def main():
    matches = load_matches(CSV_PATH)
    points = compute_standard_points(matches)
    std_rank = get_rank({k: float(v) for k, v in points.items()})

    teams, idx, records = build_directed_records(matches)

    total_pts = sum(points.values())
    n_matches = len(matches)
    n_draws = 3 * n_matches - total_pts
    print(f"Dataset: {n_matches} matches,  draws={n_draws} ({n_draws/n_matches:.1%}),  wins={n_matches-n_draws} ({(n_matches-n_draws)/n_matches:.1%})")
    print(f"GAMMA_MAX = {GAMMA_MAX}")
    print()

    LAMBDA_S = 0.01

    #1. Baseline (h=0)
    print("Fitting baseline model (h=0)...")
    s0, delta0, gamma0, _, dr0, gr0, ll0_hist = fit_strength_model(
        records, n_teams=len(teams),
        lambda_s=LAMBDA_S, fit_h=False,
        init_delta_raw=0.0, init_gamma_raw=0.0, init_h=0.0,
    )
    ll0 = max(ll0_hist)
    print(f"  delta={delta0:.4f} (raw={dr0:.4f}),  gamma={gamma0:.4f} (raw={gr0:.4f}),  best_ll={ll0:.4f}")
    print()
    
    # 2. Home advantage model
    print("Fitting home advantage model...")
    s_hat, delta_hat, gamma_hat, h_hat, dr_hat, gr_hat, ll_hist = fit_strength_model(
        records, n_teams=len(teams),
        lambda_s=LAMBDA_S, fit_h=True,
        init_delta_raw=0.0, init_gamma_raw=0.0, init_h=0.3,
    )
    best_ll = max(ll_hist)
    ll_improvement = best_ll - ll0
    print(f"  delta={delta_hat:.4f} (raw={dr_hat:.4f}),  gamma={gamma_hat:.4f} (raw={gr_hat:.4f}),  h={h_hat:.4f},  best_ll={best_ll:.4f}")
    print(f"  LL improvement over baseline: {ll_improvement:+.4f}")
    if ll_improvement > 0.5:
        print(f"  => Home advantage is a meaningful signal (+{ll_improvement:.2f} LL gain).")
    elif ll_improvement > 0:
        print(f"  => Marginal improvement — home advantage has a small but positive effect.")
    else:
        print(f"  => h did not improve fit. Home advantage is not significant in this dataset.")
    print()

    #3. Diagnostics
    print_convergence(ll_hist, best_ll)
    print_draw_probability_table(delta_hat, gamma_hat, h_hat)
    print_home_analysis(matches, teams, s_hat, delta_hat, gamma_hat, h_hat)

    #4. Latent strength ranking
    print_strength_table(teams, s_hat, points, std_rank,
                         title="Latent strength ranking (home advantage model)")
    
    # 5. Rank correlations
    str_rank   = get_rank({teams[i]: s_hat[i] for i in range(len(teams))})
    str_rank_0 = get_rank({teams[i]: s0[i] for i in range(len(teams))})

    print("=" * 56)
    print("Rank correlation summary (vs standard points table)")
    print("=" * 56)
    print(f"  Latent strength s_hat:        tau={kendall_tau(std_rank, str_rank):+.4f},  rho={spearman_rho(std_rank, str_rank):+.4f}")
    print()
    print("Baseline (h=0) vs home advantage model — strength ranking shift")
    print(f"  tau={kendall_tau(str_rank_0, str_rank):+.4f},  rho={spearman_rho(str_rank_0, str_rank):+.4f}")
    print()

    # 6. Top-k overlap
    print("Top-k overlap with standard points table")
    print(f"  Top  8 (auto qualification): {topk_overlap(std_rank, str_rank,  8)} / 8")
    print(f"  Top 24 (play-off cut):       {topk_overlap(std_rank, str_rank, 24)} / 24")
    print()


if __name__ == "__main__":
    main()

SyntaxError: invalid syntax (301284300.py, line 8)